# Chapter 2: Working with Text Data

## 2.2 Tokenizing text

In [211]:
import re
import os
import urllib.request

In [212]:
if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

In [213]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [214]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

In [215]:
class SimpleTokenizerV1:
    def __init__(self, vocab: dict[str, int]):
        self.str_to_int: dict[str, int] = vocab
        self.int_to_str: dict[int, str] = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int.get(s, -1) for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str.get(i, "<|unk|>") for i in ids])
        # Replace spaces before punctuation
        text = re.sub(r'\s+([,.:;?_!()\'])', r'\1', text)

        # Remove space after an opening quotation mark
        text = re.sub(r'"\s+', '"', text)

        # Remove space before a closing quotation mark
        # when followed by --, punctuation, or end of text
        text = re.sub(r'\s+"(?=\s*(?:--|[,.:;?!]|$))', '"', text)

        # Ensure -- has a space on both sides
        text = re.sub(r'\s*--\s*', '--', text)

        return text

## 2.3 Converting tokens in token IDs

In [216]:
all_words = sorted(set(preprocessed))

In [217]:
vocab = {token:integer for integer, token in enumerate(all_words)}
vocab_size = len(vocab)

tokenizer = SimpleTokenizerV1(vocab)

In [218]:
text = """Yes--I could just manage to see it--the first portrait of Jack's I had ever had to strain my eyes over! Usually they had the place of honour--say the central panel in a pale yellow or _rose Dubarry_ drawing-room, or a monumental easel placed so that it took the light through curtains of old Venetian point. The more modest place became the picture better; yet, as my eyes grew accustomed to the half-light, all the characteristic qualities came out--all the hesitations disguised as audacities, the tricks of prestidigitation by which, with such consummate skill, he managed to divert attention from the real business of the picture to some pretty irrelevance of detail. Mrs. Gisburn, presenting a neutral surface to work on--forming, as it were, so inevitably the background of her own picture--had lent herself in an unusual degree to the display of this false virtuosity. The picture was one of Jack's "strongest," as his admirers would have put it--it represented, on his part, a swelling of muscles, a congesting of veins, a balancing, straddling and straining, that reminded one of the circus-clown's ironic efforts to lift a feather. It met, in short, at every point the demand of lovely woman to be painted "strongly" because she was tired of being painted "sweetly"--and yet not to lose an atom of the sweetness."""
res = tokenizer.encode(text)
print(res)

[112, 6, 53, 292, 590, 657, 1016, 862, 585, 6, 988, 442, 782, 722, 57, 2, 850, 53, 514, 401, 514, 1016, 933, 697, 417, 741, 0, 103, 994, 514, 988, 773, 722, 553, 6, 856, 988, 251, 752, 568, 115, 749, 1124, 734, 114, 846, 32, 114, 365, 5, 734, 115, 682, 374, 774, 908, 987, 585, 1021, 988, 626, 1007, 309, 722, 726, 104, 779, 7, 93, 685, 678, 773, 204, 988, 769, 217, 9, 1125, 5, 177, 697, 417, 511, 126, 1016, 988, 518, 5, 145, 988, 254, 807, 243, 738, 6, 145, 988, 543, 349, 177, 186, 5, 988, 1032, 722, 789, 241, 1093, 5, 1108, 949, 287, 900, 5, 533, 658, 1016, 354, 184, 477, 988, 820, 238, 722, 988, 769, 1016, 910, 790, 582, 722, 333, 7, 67, 7, 38, 5, 788, 115, 706, 960, 1016, 1117, 727, 6, 466, 5, 177, 585, 1088, 5, 908, 572, 988, 192, 722, 539, 742, 769, 6, 514, 619, 542, 568, 156, 1050, 322, 1016, 988, 350, 722, 999, 426, 1068, 7, 93, 769, 1077, 729, 722, 57, 2, 850, 1, 941, 5, 1, 177, 549, 131, 1120, 530, 806, 585, 6, 585, 835, 5, 727, 549, 756, 5, 115, 967, 722, 695, 5, 115, 284, 722

In [219]:
t = tokenizer.decode(res)
print(t)

Yes--I could just manage to see it--the first portrait of Jack' s I had ever had to strain my eyes over! Usually they had the place of honour--say the central panel in a pale yellow or_ rose Dubarry_ drawing-room, or a monumental easel placed so that it took the light through curtains of old Venetian point. The more modest place became the picture better; yet, as my eyes grew accustomed to the half-light, all the characteristic qualities came out--all the hesitations disguised as audacities, the tricks of prestidigitation by which, with such consummate skill, he managed to divert attention from the real business of the picture to some pretty irrelevance of detail. Mrs. Gisburn, presenting a neutral surface to work on--forming, as it were, so inevitably the background of her own picture--had lent herself in an unusual degree to the display of this false virtuosity. The picture was one of Jack' s "strongest, "as his admirers would have put it--it represented, on his part, a swelling of m